## 🗺️ Parte 5: Feature Engineering Geoespacial

### Por qué Features Espaciales Mejoran Modelos

Las **características espaciales** capturan patrones que variables tradicionales no ven:

- 📍 **Distancia a puntos de interés**: Clientes cerca de sucursales compran más frecuentemente
- 🏚️ **Densidad de zona**: Áreas urbanas vs. suburbanas tienen comportamientos distintos
- 📊 **Métricas de vecindario**: El contexto espacial influye en decisiones de compra
- 🔄 **Features temporales-espaciales**: Patrones de compra por zona y hora/día

### Features Espaciales a Crear

1. **Distancia a sucursal más cercana**
2. **Densidad de clientes en zona**
3. **Facturación promedio de la zona**
4. **Número de sucursales en radio de 5km**
5. **Ventas del vecindario (hexágonos adyacentes)**

---

In [0]:
# Importar librerías
import pandas as pd
import numpy as np
import h3
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Cargar datasets
ruta_datos = '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/'

df_clientes = pd.read_csv(ruta_datos + 'clientes.csv')
df_ventas = pd.read_csv(ruta_datos + 'ventas.csv')
df_sucursales = pd.read_csv(ruta_datos + 'sucursales.csv')
df_detalles = pd.read_csv(ruta_datos + 'detalles_ventas.csv')

print("✅ Datasets cargados")
print(f"   Clientes: {len(df_clientes)} con índices H3")
print(f"   Sucursales: {len(df_sucursales)} con coordenadas")

In [0]:
print("=" * 80)
print("FEATURE 1: DISTANCIA A SUCURSAL MÁS CERCANA")
print("=" * 80)

# Obtener índices H3 de sucursales
sucursales_h3 = df_sucursales['h3_index'].tolist()

# Calcular distancia mínima para cada cliente
def calcular_distancia_min(cliente_h3, sucursales_h3):
    if pd.isna(cliente_h3):
        return None
    try:
        distancias = [h3.grid_distance(cliente_h3, s) for s in sucursales_h3]
        return min(distancias)
    except:
        return None

df_clientes['dist_sucursal_min'] = df_clientes['h3_index'].apply(
    lambda x: calcular_distancia_min(x, sucursales_h3)
)

print(f"\n📊 Estadísticas de Distancia:")
print(f"   Clientes con distancia calculada: {df_clientes['dist_sucursal_min'].notna().sum()}")
print(f"   Distancia promedio: {df_clientes['dist_sucursal_min'].mean():.2f} hexágonos")
print(f"   Distancia mínima: {df_clientes['dist_sucursal_min'].min():.0f} hex")
print(f"   Distancia máxima: {df_clientes['dist_sucursal_min'].max():.0f} hex")

# Clasificar por distancia
def clasificar_distancia(dist):
    if pd.isna(dist):
        return 'Desconocido'
    elif dist <= 2:
        return 'Muy Cerca'
    elif dist <= 5:
        return 'Cerca'
    elif dist <= 10:
        return 'Medio'
    else:
        return 'Lejos'

df_clientes['categoria_distancia'] = df_clientes['dist_sucursal_min'].apply(clasificar_distancia)

print(f"\n📍 Distribución de Clientes por Distancia:")
print(df_clientes['categoria_distancia'].value_counts())

print("\n✅ Feature creada: 'dist_sucursal_min'")

In [0]:
print("\n" + "=" * 80)
print("FEATURES 2 & 3: DENSIDAD Y FACTURACIÓN DE ZONA")
print("=" * 80)

# Feature 2: Densidad de clientes en zona
densidad_zona = df_clientes.groupby('h3_index').size().reset_index(name='densidad_zona')
df_clientes = df_clientes.merge(densidad_zona, on='h3_index', how='left')

print(f"\n📊 Feature 2 - Densidad de Zona:")
print(f"   Densidad promedio: {df_clientes['densidad_zona'].mean():.2f} clientes/zona")
print(f"   Zona más densa: {df_clientes['densidad_zona'].max()} clientes")

# Feature 3: Facturación promedio de la zona
ventas_geo = df_ventas[df_ventas['cliente_id'].notna()].merge(
    df_clientes[['cliente_id', 'h3_index']], 
    on='cliente_id'
)

facturacion_zona = ventas_geo.groupby('h3_index')['total'].agg(['mean', 'sum']).reset_index()
facturacion_zona.columns = ['h3_index', 'facturacion_promedio_zona', 'facturacion_total_zona']

df_clientes = df_clientes.merge(facturacion_zona, on='h3_index', how='left')

print(f"\n💰 Feature 3 - Facturación de Zona:")
print(f"   Facturación promedio por zona: ${df_clientes['facturacion_promedio_zona'].mean():,.0f}")
print(f"   Zona más rentable: ${df_clientes['facturacion_promedio_zona'].max():,.0f}")

# Llenar NaN (zonas sin ventas)
df_clientes['facturacion_promedio_zona'] = df_clientes['facturacion_promedio_zona'].fillna(0)
df_clientes['facturacion_total_zona'] = df_clientes['facturacion_total_zona'].fillna(0)

print("\n✅ Features creadas: 'densidad_zona', 'facturacion_promedio_zona'")

In [0]:
print("\n" + "=" * 80)
print("FEATURES 4 & 5: SUCURSALES CERCANAS Y VENTAS DE VECINDARIO")
print("=" * 80)

# Feature 4: Número de sucursales en radio de 10 hexágonos (~1km)
def contar_sucursales_cercanas(cliente_h3, sucursales_h3, radio=10):
    if pd.isna(cliente_h3):
        return 0
    try:
        return sum([1 for s in sucursales_h3 if h3.grid_distance(cliente_h3, s) <= radio])
    except:
        return 0

df_clientes['num_sucursales_cercanas'] = df_clientes['h3_index'].apply(
    lambda x: contar_sucursales_cercanas(x, sucursales_h3, radio=10)
)

print(f"\n🏪 Feature 4 - Sucursales Cercanas:")
print(df_clientes['num_sucursales_cercanas'].value_counts().sort_index())

# Feature 5: Ventas en vecindario (radio 2)
def calcular_ventas_vecindario(h3_index, ventas_geo_dict, radio=2):
    if pd.isna(h3_index):
        return 0
    try:
        # Obtener hexágonos vecinos
        vecinos = h3.grid_disk(h3_index, radio)
        # Sumar ventas de vecinos
        ventas = sum([ventas_geo_dict.get(v, 0) for v in vecinos])
        return ventas
    except:
        return 0

# Crear diccionario de ventas por zona
ventas_geo_dict = facturacion_zona.set_index('h3_index')['facturacion_total_zona'].to_dict()

df_clientes['ventas_vecindario'] = df_clientes['h3_index'].apply(
    lambda x: calcular_ventas_vecindario(x, ventas_geo_dict, radio=2)
)

print(f"\n📋 Feature 5 - Ventas de Vecindario:")
print(f"   Promedio: ${df_clientes['ventas_vecindario'].mean():,.0f}")
print(f"   Máximo: ${df_clientes['ventas_vecindario'].max():,.0f}")

print("\n✅ Features creadas: 'num_sucursales_cercanas', 'ventas_vecindario'")

## 🔄 Parte 6: Detección y Monitoreo de Data Drift

### ¿Qué es Data Drift?

**Data Drift** (deriva de datos) ocurre cuando la distribución de los datos de entrada cambia con el tiempo, causando que modelos entrenados pierdan precisión.

### Tipos de Drift

1. **Concept Drift** 🧠
   - La relación entre features y target cambia
   - Ejemplo: "Productos saludables" antes no vendían, ahora sí

2. **Data Drift (Covariate Shift)** 📊
   - La distribución de features cambia
   - Ejemplo: Más clientes en zona Este que antes

3. **Label Drift** 🎯
   - La distribución del target cambia
   - Ejemplo: Ticket promedio sube por inflación

### Por qué Monitorear Drift

❌ **Sin monitoreo**: Modelo se degrada silenciosamente  
✅ **Con monitoreo**: Detectar cuándo re-entrenar modelo

### Métodos de Detección

1. **Estadísticos**: Comparar estadísticas (media, std, quantiles)
2. **Pruebas de hipótesis**: Kolmogorov-Smirnov, Chi-cuadrado
3. **Distancias**: KL-divergence, Wasserstein distance
4. **Modelos**: Entrenar clasificador train vs. production data

---

In [0]:
print("=" * 80)
print("SIMULACIÓN DE DATA DRIFT")
print("=" * 80)

# Simular dos períodos: 2024 (baseline) vs. 2025 (actual)
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
df_ventas['anio'] = df_ventas['fecha'].dt.year

# Preparar datos con features espaciales
ventas_clientes = df_ventas.merge(
    df_clientes[['cliente_id', 'h3_index', 'dist_sucursal_min', 'densidad_zona', 'facturacion_promedio_zona']], 
    on='cliente_id',
    how='left'
)

# Filtrar solo registros con features
ventas_clientes = ventas_clientes.dropna(subset=['dist_sucursal_min'])

# Dividir por año
df_2024 = ventas_clientes[ventas_clientes['anio'] == 2024]
df_2025 = ventas_clientes[ventas_clientes['anio'] == 2025]

print(f"\n📊 Datos por período:")
print(f"   2024 (baseline): {len(df_2024):,} ventas")
print(f"   2025 (actual): {len(df_2025):,} ventas")

In [0]:
print("\n" + "=" * 80)
print("MÉTODO 1: COMPARACIÓN DE ESTADÍSTICAS")
print("=" * 80)

# Features a monitorear
features = ['total', 'dist_sucursal_min', 'densidad_zona', 'facturacion_promedio_zona']

print("\n🔍 Comparación de Distribuciones (2024 vs. 2025):\n")

drift_report = []

for feature in features:
    stats_2024 = df_2024[feature].describe()
    stats_2025 = df_2025[feature].describe()
    
    mean_change = ((stats_2025['mean'] - stats_2024['mean']) / stats_2024['mean']) * 100
    std_change = ((stats_2025['std'] - stats_2024['std']) / stats_2024['std']) * 100
    
    # Detectar drift (cambio > 10%)
    drift_detected = abs(mean_change) > 10 or abs(std_change) > 10
    
    drift_report.append({
        'feature': feature,
        'mean_2024': stats_2024['mean'],
        'mean_2025': stats_2025['mean'],
        'mean_change_%': mean_change,
        'std_change_%': std_change,
        'drift': '⚠️ Sí' if drift_detected else '✅ No'
    })
    
    print(f"{feature}:")
    print(f"   Media 2024: {stats_2024['mean']:.2f}")
    print(f"   Media 2025: {stats_2025['mean']:.2f}")
    print(f"   Cambio: {mean_change:+.2f}%")
    if drift_detected:
        print(f"   ⚠️ DRIFT DETECTADO")
    print()

df_drift_report = pd.DataFrame(drift_report)
print("\n📊 Resumen de Drift:")
print(df_drift_report)

In [0]:
print("\n" + "=" * 80)
print("MÉTODO 2: TEST DE KOLMOGOROV-SMIRNOV")
print("=" * 80)

from scipy.stats import ks_2samp

print("\n🧪 Test KS: Compara si dos muestras vienen de la misma distribución")
print("   p-value < 0.05 → Distribuciones son DIFERENTES (drift)\n")

ks_results = []

for feature in features:
    # Aplicar test KS
    statistic, p_value = ks_2samp(
        df_2024[feature].dropna(), 
        df_2025[feature].dropna()
    )
    
    drift_detected = p_value < 0.05
    
    ks_results.append({
        'feature': feature,
        'ks_statistic': statistic,
        'p_value': p_value,
        'drift': '⚠️ DRIFT' if drift_detected else '✅ OK'
    })
    
    print(f"{feature}:")
    print(f"   KS statistic: {statistic:.4f}")
    print(f"   p-value: {p_value:.4f}")
    if drift_detected:
        print(f"   ⚠️ DRIFT SIGNIFICATIVO DETECTADO")
    else:
        print(f"   ✅ No hay evidencia de drift")
    print()

df_ks = pd.DataFrame(ks_results)
print("\n📊 Resumen KS Test:")
print(df_ks)

### 🚨 Acciones ante Detección de Drift

#### 1. **Investigar la Causa**
- ¿Cambió el negocio? (nueva sucursal, promoción, estacionalidad)
- ¿Error en datos? (bug en pipeline, datos faltantes)
- ¿Cambio real en comportamiento? (tendencia de mercado)

#### 2. **Re-entrenar Modelo**
- Usar datos más recientes
- Ajustar hiperparámetros
- Considerar features adicionales

#### 3. **Monitoreo Continuo**
- Establecer alertas automáticas
- Dashboard de drift en producción
- Re-entrenamiento programado (ej. mensual)

#### 4. **Versionado**
- Guardar versiones de modelos
- Rollback si nuevo modelo falla
- A/B testing modelo viejo vs. nuevo

---

### 🛠️ Herramientas para Drift Monitoring

- **Evidently AI**: Reportes de drift automáticos
- **MLflow**: Tracking de métricas en producción
- **Great Expectations**: Validación de calidad de datos
- **Databricks Lakehouse Monitoring**: Built-in en Databricks

---

# TP06: Feature Engineering y Modelado
## Laboratorio (Herramientas) - Universidad del Aconcagua
### Unidad 3: Herramientas en la Nube de Modelado de Datos

---

### 🎯 Objetivos del Trabajo Práctico

1. Realizar **ingeniería de características** (feature engineering)
2. **Estandarizar variables numéricas**
3. Entrenar un **modelo predictivo** con Scikit-learn
4. Evaluar el **desempeño del modelo**
5. Interpretar **resultados y predicciones**

---

### 📁 Caso de Estudio: Predicción de Ventas

Desarrollaremos un modelo para predecir las ventas futuras de la panadería.

### 🕰️ Duración Estimada: 3 horas

In [0]:
# Importar librerías
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Librerías importadas")
print("Listo para feature engineering y modelado")

## Parte 1: Preparar Dataset para Modelado

### 📂 Consolidar datos para predicción

Vamos a crear un dataset consolidado que incluya features relevantes para predecir la facturación diaria.

In [0]:
# Cargar datos
ruta_datos = '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/'

df_ventas = pd.read_csv(ruta_datos + 'ventas.csv', parse_dates=['fecha'])
df_detalles = pd.read_csv(ruta_datos + 'detalles_ventas.csv')
df_productos = pd.read_csv(ruta_datos + 'productos.csv')
df_sucursales = pd.read_csv(ruta_datos + 'sucursales.csv')

# Unir para crear dataset completo
df_completo = df_detalles.merge(df_ventas[['venta_id', 'fecha', 'sucursal_id']], on='venta_id')
df_completo = df_completo.merge(df_productos[['producto_id', 'categoria']], on='producto_id')
df_completo = df_completo.merge(df_sucursales[['sucursal_id', 'zona']], on='sucursal_id')

print(f"✅ Dataset consolidado: {len(df_completo):,} registros")
print(f"\n📌 Columnas disponibles: {list(df_completo.columns)}")
df_completo.head()

In [0]:
# Agregar ventas por día para crear el dataset de entrenamiento
# Objetivo: predecir la facturación total de un día

df_diario = df_completo.groupby(['fecha', 'sucursal_id', 'zona']).agg({
    'venta_id': 'nunique',  # número de transacciones
    'subtotal': 'sum'  # facturación total
}).rename(columns={
    'venta_id': 'numero_ventas',
    'subtotal': 'facturacion_total'
}).reset_index()

print(f"✅ Dataset diario creado: {len(df_diario):,} registros")
print(f"\n📊 Primeras filas:")
display(df_diario.head(10))

## Parte 2: Ingeniería de Características (Feature Engineering)

### 🏭 Crear features predictivas

Vamos a crear características que ayuden a predecir la facturación: día de semana, mes, tendencias, etc.

In [0]:
# Crear features temporales
df_diario['fecha'] = pd.to_datetime(df_diario['fecha'])

# Features de tiempo
df_diario['anio'] = df_diario['fecha'].dt.year
df_diario['mes'] = df_diario['fecha'].dt.month
df_diario['dia'] = df_diario['fecha'].dt.day
df_diario['dia_semana'] = df_diario['fecha'].dt.dayofweek  # 0=Lunes, 6=Domingo
df_diario['es_fin_semana'] = (df_diario['dia_semana'] >= 5).astype(int)
df_diario['trimestre'] = df_diario['fecha'].dt.quarter
df_diario['semana_del_anio'] = df_diario['fecha'].dt.isocalendar().week

# Feature de días desde inicio
df_diario['dias_desde_inicio'] = (df_diario['fecha'] - df_diario['fecha'].min()).dt.days

print("✅ Features temporales creadas")
print(f"\n📊 Nuevas columnas:")
print([col for col in df_diario.columns if col not in ['fecha', 'sucursal_id', 'zona', 'numero_ventas', 'facturacion_total']])

In [0]:
# Crear features de tendencia (promedios móviles)
df_diario = df_diario.sort_values(['sucursal_id', 'fecha'])

# Promedio móvil de 7 días (semana anterior)
df_diario['facturacion_ma7'] = df_diario.groupby('sucursal_id')['facturacion_total'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

# Promedio móvil de 30 días (mes anterior)
df_diario['facturacion_ma30'] = df_diario.groupby('sucursal_id')['facturacion_total'].transform(
    lambda x: x.rolling(window=30, min_periods=1).mean()
)

# Diferencia con día anterior (lag)
df_diario['facturacion_lag1'] = df_diario.groupby('sucursal_id')['facturacion_total'].shift(1)
df_diario['facturacion_diff'] = df_diario['facturacion_total'] - df_diario['facturacion_lag1']

print("✅ Features de tendencia creadas")
print("\n📊 Muestra con nuevas features:")
display(df_diario[['fecha', 'sucursal_id', 'facturacion_total', 'facturacion_ma7', 'facturacion_ma30']].head(10))

## Parte 3: Encoding y Escalado de Variables

### 🔢 Transformar variables para ML

Los modelos de machine learning requieren que todas las variables sean numéricas y estén en escalas similares.

In [0]:
# Encoding de variables categóricas con LabelEncoder
le_zona = LabelEncoder()
df_diario['zona_encoded'] = le_zona.fit_transform(df_diario['zona'])

le_sucursal = LabelEncoder()
df_diario['sucursal_encoded'] = le_sucursal.fit_transform(df_diario['sucursal_id'])

print("✅ Variables categóricas codificadas")
print("\n📊 Mapeos:")
print("Zonas:", dict(zip(le_zona.classes_, le_zona.transform(le_zona.classes_))))
print("Sucursales:", dict(zip(le_sucursal.classes_, le_sucursal.transform(le_sucursal.classes_))))

In [0]:
# Eliminar filas con valores nulos (primeros días sin lags)
df_modelo = df_diario.dropna().copy()

# Seleccionar features para el modelo
features = [
    'sucursal_encoded', 'zona_encoded',
    'anio', 'mes', 'dia', 'dia_semana', 'es_fin_semana', 'trimestre', 'semana_del_anio',
    'dias_desde_inicio', 'numero_ventas',
    'facturacion_ma7', 'facturacion_ma30', 'facturacion_lag1'
]

target = 'facturacion_total'

X = df_modelo[features]
y = df_modelo[target]

print(f"✅ Dataset preparado para modelado")
print(f"\n📊 Dimensiones:")
print(f"  Features (X): {X.shape}")
print(f"  Target (y): {y.shape}")
print(f"\n📌 Features seleccionadas:")
print(features)

In [0]:
# Dividir en train y test (80% - 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=False  # shuffle=False para respetar orden temporal
)

print(f"✅ Dataset dividido")
print(f"  Train: {X_train.shape[0]:,} registros")
print(f"  Test: {X_test.shape[0]:,} registros")

# Escalar features numéricas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Features escaladas con StandardScaler")

## Parte 4: Entrenamiento del Modelo Predictivo

### 🤖 Random Forest Regressor

Usaremos Random Forest, un algoritmo robusto que maneja bien relaciones no lineales y variables categóricas.

In [0]:
# Entrenar modelo Random Forest
print("🤖 Entrenando modelo Random Forest...")

model = RandomForestRegressor(
    n_estimators=100,  # número de árboles
    max_depth=10,      # profundidad máxima
    min_samples_split=5,
    random_state=42,
    n_jobs=-1  # usar todos los cores
)

model.fit(X_train_scaled, y_train)

print("✅ Modelo entrenado exitosamente")
print(f"\n🌳 Número de árboles: {model.n_estimators}")
print(f"🌳 Profundidad máxima: {model.max_depth}")

In [0]:
# Realizar predicciones
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

print("✅ Predicciones realizadas")
print(f"\n📄 Comparación (primeras 10 predicciones del test):")

comparacion = pd.DataFrame({
    'Real': y_test.values[:10],
    'Predicho': y_test_pred[:10],
    'Diferencia': y_test.values[:10] - y_test_pred[:10],
    'Error_%': np.abs((y_test.values[:10] - y_test_pred[:10]) / y_test.values[:10] * 100).round(2)
})

display(comparacion)

In [0]:
# Calcular métricas de evaluación
mae_train = mean_absolute_error(y_train, y_train_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)

rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print("📊 MÉTRICAS DE EVALUACIÓN")
print("=" * 80)
print(f"\nTrain Set:")
print(f"  MAE (Mean Absolute Error):  ${mae_train:,.2f}")
print(f"  RMSE (Root Mean Squared Error): ${rmse_train:,.2f}")
print(f"  R² Score: {r2_train:.4f}")

print(f"\nTest Set:")
print(f"  MAE (Mean Absolute Error):  ${mae_test:,.2f}")
print(f"  RMSE (Root Mean Squared Error): ${rmse_test:,.2f}")
print(f"  R² Score: {r2_test:.4f}")

print("\n" + "=" * 80)
print("\n💡 Interpretación:")
print(f"  - El modelo predice con un error promedio de ${mae_test:,.2f}")
print(f"  - Explica el {r2_test*100:.1f}% de la varianza en las ventas")

In [0]:
# Visualizar predicciones vs valores reales
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Scatter plot: Real vs Predicho
ax1.scatter(y_test, y_test_pred, alpha=0.5, s=20)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect prediction')
ax1.set_xlabel('Facturación Real ($)', fontsize=12)
ax1.set_ylabel('Facturación Predicha ($)', fontsize=12)
ax1.set_title('🎯 Predicciones vs Valores Reales', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Distribución de errores
errores = y_test - y_test_pred
ax2.hist(errores, bins=50, edgecolor='black', alpha=0.7, color='coral')
ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Error = 0')
ax2.set_xlabel('Error de Predicción ($)', fontsize=12)
ax2.set_ylabel('Frecuencia', fontsize=12)
ax2.set_title('📉 Distribución de Errores', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# Analizar importancia de features
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("🎯 IMPORTANCIA DE FEATURES")
print("=" * 80)
display(feature_importance)

# Visualizar top 10 features más importantes
fig, ax = plt.subplots(figsize=(10, 6))
top_features = feature_importance.head(10)
ax.barh(range(len(top_features)), top_features['importance'], color='steelblue', edgecolor='black')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'])
ax.set_xlabel('Importancia', fontsize=12)
ax.set_title('🏆 Top 10 Features Más Importantes', fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 🎯 Resumen del TP06

### ✅ Qué aprendimos:

1. **Feature Engineering**: Creamos features temporales, de tendencia y lags
2. **Encoding**: Transformamos variables categóricas con LabelEncoder
3. **Escalado**: Normalizamos features con StandardScaler
4. **Modelado**: Entrenamos un Random Forest Regressor
5. **Evaluación**: Calculamos MAE, RMSE y R² Score
6. **Interpretación**: Analizamos importancia de features y errores

### 📊 Resultados del modelo:

* MAE: Error promedio en pesos
* RMSE: Penaliza errores grandes
* R²: Porcentaje de varianza explicada
* Features más importantes: promedios móviles, lags, día de semana

### 🚀 Próximos pasos:

En la **Unidad 4** (Proyectos Integradores) aprenderemos a:
* Crear pipelines completos de datos
* Automatizar procesos de ETL
* Orquestar workflows complejos
* Desarrollar proyectos end-to-end

---

**📝 Excelente! Has completado la Unidad 3 - Modelado de Datos.**

### 📚 UNIDADES 1, 2 Y 3 COMPLETADAS ✅

* **Unidad 1 - Análisis de Datos**: Carga, transformación y exploración
* **Unidad 2 - Visualización de Datos**: Perfilado y dashboards
* **Unidad 3 - Modelado de Datos**: Estructuración y feature engineering

Solo falta la **Unidad 4 - Proyectos Integradores** para completar el curso!